In [23]:
import pulser
import numpy as np
import torch

import emu_base 
import emu_mps

import torch
import numpy as np
from sequence_2_matrix import A_direct_mat, b_direct_vec, solve_cd_torch

In [24]:
Omega_max = 2 * 2 * np.pi
delta_0 = -6 * Omega_max / 2
delta_f = 1 * Omega_max / 2
t_rise = 200
t_fall = 500
sweep_factor = 1

In [25]:
num_atoms = 5
reg = pulser.Register.rectangle(1, num_atoms, prefix="q",spacing=torch.tensor(7.0,requires_grad=True))

In [26]:
t_sweep = (delta_f - delta_0) / (2 * np.pi * 10) * 1000 * sweep_factor
rise = pulser.Pulse.ConstantDetuning(
    pulser.waveforms.RampWaveform(t_rise, 0.0, Omega_max), delta_0, 0.0
)
sweep = pulser.Pulse.ConstantAmplitude(
    Omega_max, pulser.waveforms.RampWaveform(int(t_sweep), delta_0, delta_f), 0.0
)
fall = pulser.Pulse.ConstantDetuning(
    pulser.waveforms.RampWaveform(t_fall, Omega_max, 0.0), delta_f, 0.0
)
seq = pulser.Sequence(reg, pulser.MockDevice)
seq.declare_channel("ising_global", "rydberg_global")
seq.add(rise, "ising_global")
seq.add(sweep, "ising_global")
seq.add(fall, "ising_global")

In [27]:
dt = 100
emu_mps_config = emu_mps.MPSConfig()
pulser_data = emu_base.pulser_adapter.PulserData(sequence=seq,config=emu_mps_config,dt=dt)

/home/mauro/Documents/pasqal/code/emulators/emu_mps/mps_config.py:111: UserWarning: 'MPSConfig' was initialized without any observables. The corresponding emulation results will be empty.
  super().__init__(


In [28]:
sequence_data = pulser_data.get_sequences()
omegas = [i.omega.to(dtype=torch.float64).requires_grad_(True) for i in sequence_data]
sequence_data = pulser_data.get_sequences()
deltas = [i.delta.to(dtype=torch.float64).requires_grad_(True) for i in sequence_data]
sequence_data = pulser_data.get_sequences()
sequence_data = pulser_data.get_sequences()
phis = [i.phi.to(dtype=torch.float64).requires_grad_(True) for i in sequence_data]
# sequence_data = pulser_data.get_sequences()
# # Get matrix at a specific time t (float, in ns)
# interact = [i.interaction_matrix(0.0) for i in sequence_data]
# sequence_data = pulser_data.get_sequences()
# # Or get both explicitly
# interact_masked = [i.interaction_matrix.masked_matrix for i in sequence_data]
sequence_data = pulser_data.get_sequences()
interact_full   = [i.interaction_matrix.full_matrix   for i in sequence_data]

In [29]:
interact_full

[tensor([[0.0000e+00, 4.6071e+01, 7.1985e-01, 6.3197e-02, 1.1248e-02],
         [4.6071e+01, 0.0000e+00, 4.6071e+01, 7.1985e-01, 6.3197e-02],
         [7.1985e-01, 4.6071e+01, 0.0000e+00, 4.6071e+01, 7.1985e-01],
         [6.3197e-02, 7.1985e-01, 4.6071e+01, 0.0000e+00, 4.6071e+01],
         [1.1248e-02, 6.3197e-02, 7.1985e-01, 4.6071e+01, 0.0000e+00]],
        dtype=torch.float64, grad_fn=<IndexPutBackward0>)]

In [30]:
# ── 1. Setup ─────────────────────────────────────────────────────────────────
n_atoms = num_atoms
N = 15  # number of time steps
dt = 50.0  # ns per step



# ── 2. Boundary conditions (fixed, not optimized) ─────────────────────────
Omega_max = 2 * 2 * np.pi
delta_0 = -6 * Omega_max / 2
delta_f = 1 * Omega_max / 2

bc_omega = torch.tensor([0.0, 0.0], dtype=torch.float64)  # [start, end]
bc_delta = torch.tensor([delta_0, delta_f], dtype=torch.float64)
bc_phi = torch.tensor([0.0, 0.0], dtype=torch.float64)

# ── 3. Free interior parameters (what we optimize) ────────────────────────
# Shape: (N-1,) — N+1 total points, 2 fixed at boundaries
omega_free = torch.linspace(0.0, Omega_max, N - 1, dtype=torch.float64).requires_grad_(
    True
)
delta_free = torch.linspace(
    delta_0, delta_f, N - 1, dtype=torch.float64
).requires_grad_(True)
phi_free = torch.zeros(N - 1, dtype=torch.float64).requires_grad_(True)


def full_pulse():
    """Assemble full pulse [start, interior..., end] — BCs are fixed."""
    omega = torch.cat([bc_omega[:1], omega_free, bc_omega[1:]])  # shape (N+1,)
    delta = torch.cat([bc_delta[:1], delta_free, bc_delta[1:]])
    phi = torch.cat([bc_phi[:1], phi_free, bc_phi[1:]])
    return omega, delta, phi


# Sample the original sequence at interior time points
# original_omega = torch.tensor([seq_omega_at(k * dt) for k in range(1, N)], dtype=torch.float64)
# omega_free = original_omega.detach().clone().requires_grad_(True)


# ── 5. Time derivatives (user-provided or finite difference) ─────────────
def compute_derivatives(omegas, deltas, phis):
    """
    Central differences for interior, forward/backward at boundaries.
    Returns dOmega_H, dMu_H, dNu_H of same shape.
    Replace with your own derivative tensors if you have.
    """
    omegas_t = omegas[0].real  # (N, n_atoms), float64
    deltas_t = deltas[0].real
    phis_t   = phis[0].real

    Omega_H = omegas_t / 2 * torch.cos(phis_t)
    Mu_H    = omegas_t / 2 * torch.sin(phis_t)
    Nu_H    = deltas_t

    def grad(x):
        d_start    = (x[1:2]   - x[0:1])   / dt
        d_interior = (x[2:]    - x[:-2])   / (2 * dt)
        d_end      = (x[-1:]   - x[-2:-1]) / dt
        return torch.cat([d_start, d_interior, d_end], dim=0)

    return grad(Omega_H), grad(Mu_H), grad(Nu_H)


test_omega, test_mu, test_nu = compute_derivatives(omegas, deltas, phis)

In [31]:
# ── 6. Cost: sum of squared CD coefficient norms over time ────────────────
def cd_cost():
    #Omega_H, Mu_H, Nu_H = pulse_to_hamiltonian(omega, delta, phi)
    # Omega_H = torch.cat([bc_OH[:1], Omega_H_free, bc_OH[1:]])   # (N+1, n_atoms)
    # Mu_H    = torch.cat([bc_MH[:1], Mu_H_free,    bc_MH[1:]])
    # Nu_H    = torch.cat([bc_NH[:1], Nu_H_free,    bc_NH[1:]])
    dOmega_H, dMu_H, dNu_H = compute_derivatives(omegas, deltas, phis)
    n_single = 3*n_atoms  # number of single-body terms (Omega, Mu, Nu)
    total_cost = torch.tensor(0.0, dtype=torch.float64)
    for k in range(len(omegas[0])):
        M = A_direct_mat(n_atoms, omegas[0][k], deltas[0][k], phis[0][k], interact_full[0])
        b = b_direct_vec(n_atoms, dOmega_H[k], dMu_H[k], dNu_H[k])
        coeffs = solve_cd_torch(M, b)         # CD coefficients at step k
        two_body = coeffs[n_single:]   # symmetric + asymmetric 2-body
        total_cost = total_cost + (two_body ** 2).sum()
        
    #print(coeffs[:n_single:])
    return total_cost

# ── 7. Optimization loop ──────────────────────────────────────────────────
optimizer = torch.optim.Adam([omegas[0], deltas[0], phis[0]], lr=1e-4)

for step in range(20):
    optimizer.zero_grad()
    loss = cd_cost()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([omegas[0], deltas[0], phis[0]], max_norm=1.0)

    # after backward, check if gradient norm is actually meaningful
    # print("grad norm omega:", omegas[0].grad.norm())
    # print("grad norm delta:", deltas[0].grad.norm())
    # print("grad norm phi:", phis[0].grad.norm())

    optimizer.step()
    if step % 5 == 0:
        print(f"step {step:4d}  cost = {loss.item():.6f}")

# ── 8. Inspect result ─────────────────────────────────────────────────────
#omega_opt, delta_opt, phi_opt = full_pulse()




step    0  cost = 0.005646
step    5  cost = 0.005665
step   10  cost = 0.005646
step   15  cost = 0.005652


In [32]:
print("omega:", omegas[0].detach())
print("delta:", deltas[0].detach())
print("phi:", phis[0].detach())


omega: tensor([[ 3.1570,  3.1570,  3.1573,  3.1580,  3.1578],
        [ 9.4716,  9.4716,  9.4722,  9.4731,  9.4731],
        [12.5667, 12.5668, 12.5664, 12.5657, 12.5660],
        [12.5669, 12.5669, 12.5669, 12.5669, 12.5669],
        [12.5668, 12.5668, 12.5668, 12.5668, 12.5668],
        [12.5665, 12.5665, 12.5665, 12.5665, 12.5665],
        [12.5659, 12.5659, 12.5659, 12.5659, 12.5659],
        [12.5661, 12.5661, 12.5661, 12.5661, 12.5661],
        [12.5664, 12.5666, 12.5660, 12.5660, 12.5658],
        [11.3077, 11.3077, 11.3069, 11.3068, 11.3066],
        [ 8.7891,  8.7890,  8.7893,  8.7887,  8.7887],
        [ 6.2708,  6.2707,  6.2708,  6.2713,  6.2714],
        [ 3.7523,  3.7522,  3.7520,  3.7528,  3.7530],
        [ 1.2338,  1.2336,  1.2340,  1.2345,  1.2348]], dtype=torch.float64)
delta: tensor([[-37.6973, -37.6975, -37.6985, -37.6979, -37.6992],
        [-37.6973, -37.6985, -37.6975, -37.6981, -37.6974],
        [-34.5512, -34.5517, -34.5529, -34.5518, -34.5525],
        [-28.2